# EDA of quality-controlled anime data

## tl;dr

This notebook is a read-only consumer. Executed aggregate outputs quantify the sentinel separation, sparsity/long-tail behavior, catalogue genre/type coverage, and the distinction between interaction popularity and catalogue membership.

## Context & Methods

Only canonical Parquet tables and their audit are read. Large interaction data remains in Spark; only small aggregates are converted to pandas. No canonical processed table is modified.

### Key Assumptions

- `-1` is watched-unrated and excluded from explicit labels.
- Genre is a fallback signal, not a replacement for fair ALS evaluation.
- Popularity shown here is descriptive and must not be reused as a test-aware baseline.

In [1]:
from pathlib import Path
import json
import os
import sys
import pandas as pd
from pyspark.sql import functions as F

START_DIR = Path.cwd()
ROOT = START_DIR if (START_DIR / "src").exists() else START_DIR.parent
if not (ROOT / "src").exists() or not (ROOT / "database").exists():
    raise RuntimeError("Run this notebook from the repository root")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.config import PROCESSED_DIR, FIGURES_DIR
from src.eda import generate_figures
from src.pipeline import create_spark

required = [PROCESSED_DIR / f for f in ["clean_ratings.parquet", "clean_anime.parquet", "watched_unrated.parquet", "genre_features.parquet", "data_quality_summary.csv"]]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(f"Run notebook 01 first; missing outputs: {missing}")
spark = create_spark()

## Data

In [2]:
ratings = spark.read.parquet(str(PROCESSED_DIR / "clean_ratings.parquet"))
anime = spark.read.parquet(str(PROCESSED_DIR / "clean_anime.parquet"))
watched = spark.read.parquet(str(PROCESSED_DIR / "watched_unrated.parquet"))
audit = pd.read_csv(PROCESSED_DIR / "data_quality_summary.csv")
print({"explicit_rows": ratings.count(), "watched_unrated_pairs": watched.count(), "catalog_items": anime.count()})

{'explicit_rows': 6337232, 'watched_unrated_pairs': 1476488, 'catalog_items': 12294}


## Results

### Generate the six report-ready figures

In [3]:
figures = generate_figures(spark)
figures

{'data_quality_before_after': 'C:\\Users\\Quoc\\Desktop\\DTU\\ML Large Data\\Anime Recommend\\outputs\\figures\\data_quality_before_after.png',
 'genres_and_types': 'C:\\Users\\Quoc\\Desktop\\DTU\\ML Large Data\\Anime Recommend\\outputs\\figures\\genres_and_types.png',
 'interactions_per_anime': 'C:\\Users\\Quoc\\Desktop\\DTU\\ML Large Data\\Anime Recommend\\outputs\\figures\\interactions_per_anime.png',
 'interactions_per_user': 'C:\\Users\\Quoc\\Desktop\\DTU\\ML Large Data\\Anime Recommend\\outputs\\figures\\interactions_per_user.png',
 'rating_distribution': 'C:\\Users\\Quoc\\Desktop\\DTU\\ML Large Data\\Anime Recommend\\outputs\\figures\\rating_distribution.png',
 'top_anime': 'C:\\Users\\Quoc\\Desktop\\DTU\\ML Large Data\\Anime Recommend\\outputs\\figures\\top_anime.png'}

### Long-tail percentiles and sparsity

In [4]:
user_counts = ratings.groupBy("user_id").count()
item_counts = ratings.groupBy("anime_id").count()
user_q = user_counts.approxQuantile("count", [0.5, 0.9, 0.99], 0.001)
item_q = item_counts.approxQuantile("count", [0.5, 0.9, 0.99], 0.001)
n_users = ratings.select("user_id").distinct().count()
n_items = ratings.select("anime_id").distinct().count()
n_rows = ratings.count()
sparsity = 1 - n_rows / (n_users * n_items)
summary = {"users": n_users, "items": n_items, "explicit_rows": n_rows, "matrix_sparsity": sparsity,
           "user_interaction_p50_p90_p99": user_q, "item_interaction_p50_p90_p99": item_q}
summary

{'users': 69600,
 'items': 9926,
 'explicit_rows': 6337232,
 'matrix_sparsity': 0.9908269006741843,
 'user_interaction_p50_p90_p99': [45.0, 229.0, 616.0],
 'item_interaction_p50_p90_p99': [57.0, 1671.0, 8488.0]}

## Takeaways

- The sentinel and explicit-rating chart demonstrates why `-1` cannot be treated as zero or dislike.
- The user/item histograms and percentiles show a sparse, long-tail matrix, motivating latent-factor ALS and explicit cold-start coverage reporting.
- Genre/type coverage supports a separate content-based fallback for cold starts.
- Interaction count and catalogue members are different popularity concepts. Any evaluated baseline must be fitted on train only.
- RMSE/MAE alone cannot measure Top-N usefulness; Member 2 must also report Precision@K, Recall@K, and warm-row/candidate coverage.

In [5]:
spark.stop()